In [5]:
!pip install transformers accelerate bitsandbytes datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.9 MB/s eta 0:00:00


In [1]:
!git clone https://github.com/keshu-bara/DigitalTwin


Cloning into 'DigitalTwin'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 52 (delta 19), reused 39 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 164.48 KiB | 13.71 MiB/s, done.
Resolving deltas: 100% (19/19), done.


In [2]:
%cd DigitalTwin

/content/DigitalTwin


In [3]:
%cd notebooks/

/content/DigitalTwin/notebooks


In [4]:
!ls

01_data_exploration.ipynb  basellm_pipeline.ipynb  samplenotebook.ipynb


In [6]:
import json

In [9]:
with open("../data/processed/reply_dataset.json",'r',encoding="utf-8") as f:
  dataset = json.load(f)

# Shuffle and split

import random
random.shuffle(dataset)

split = int(0.8 * len(dataset))
train_data = dataset[:split]
test_data = dataset[split:]

len(test_data)
len(train_data)

279

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.1"


toeknizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype= torch.float16
)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [11]:
examples = train_data[:5]

persona_prompt = "You are imitating NIkhil Mishra CE.\n\n"

for ex in examples:
  persona_prompt += f"Context:\n{ex['context']}\n"
  persona_prompt += f"Reply:\n{ex['reply']}\n\n"

persona_prompt += "Now reply to the following context in same style.\n"


In [13]:
persona_prompt

'You are imitating NIkhil Mishra CE.\n\nContext:\nOkay\nOnline kyo?\nReply:\nonline he decide hua tha\n\nContext:\ntere unimob ke code repo private h?\nReply:\noffcourse\n\nContext:\nAccha\nReply:\n?\n\nContext:\nAur kssa chal raha h sbb?\nReply:\nYaar waste lag raha tha\nBadiya\n\nContext:\nI am in bro\nChalte h\nReply:\nnice !\nkarte hai phir meet pe discuss kaise chalna hai\n\nNow reply to the following context in same style.\n'

In [18]:
test_context = test_data[0]["context"]

persona_prompt = """You are imitating Nikhil Mishra CE.
Reply exactly like him — same slang, same tone, same emoji usage.
Do not repeat the format. Only give the reply text.

Examples:

Context: Okay Online kyo?
Reply: online he decide hua tha

Context: tere unimob ke code repo private h?
Reply: offcourse

Context: Accha
Reply: ?

Context: Aur kssa chal raha h sbb?
Reply: Yaar waste lag raha tha Badiya

Context: I am in bro Chalte h
Reply: nice ! karte hai phir meet pe discuss kaise chalna hai

Now respond to this new context:


Context: bhai kuch smjh nahi aa raha?
Reply:"""

final_prompt = persona_prompt

inputs = toeknizer(final_prompt, return_tensors = "pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.2
)

print(toeknizer.decode(output[0],skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


You are imitating Nikhil Mishra CE.
Reply exactly like him — same slang, same tone, same emoji usage.
Do not repeat the format. Only give the reply text.

Examples:

Context: Okay Online kyo?
Reply: online he decide hua tha

Context: tere unimob ke code repo private h?
Reply: offcourse

Context: Accha
Reply: ?

Context: Aur kssa chal raha h sbb?
Reply: Yaar waste lag raha tha Badiya

Context: I am in bro Chalte h
Reply: nice ! karte hai phir meet pe discuss kaise chalna hai

Now respond to this new context:


Context: bhai kuch smjh nahi aa raha?
Reply: nahi gaya tha
